# Training the clusterer

In [ ]:
from wordnet.SynsetClusterer import SynsetClusterer
import json

In [ ]:
# Due to the way wordnet_ic is written in NLTK, you should either put the ic file in the default directory of NLTK (nltk_data/corpora/wordnet_ic) or provide the absolute path
clusterer = SynsetClusterer(gold_cluster_file='clustering/nouns_gold_cluster_data.tsv',greek_ic_file='ic-greek-nouns.dat')

In [ ]:
cosine_similarities, cosine_index = clusterer.build_token_embeddings(clusterer.lemma_gold_clusters.keys(),batch_size=50)

In [ ]:
data = clusterer.build_training_data(cosine_similarities,cosine_index)

In [ ]:
# By default, we train 10-fold. Ignore accuracy at the end: it is not properly implemented when we return probabilities.
classifier = clusterer.train_classifier(data)

In [ ]:
best_threshold = clusterer.get_optimal_threshold(classifier)

In [ ]:
classifier = clusterer.train_classifier(data,nfold=False)
print(clusterer.cluster_similarity_model.intercept_[0])
for feature, coef in zip(clusterer.cluster_similarity_model.feature_names_in_,clusterer.cluster_similarity_model.coef_[0]):
    print(feature,coef)

In [ ]:
# We train a new model where all features with 0 coefficients ignored, so we don't need to compute features that we're not using
classifier = clusterer.train_classifier(data,nfold=False,ignore_columns=['LEMMA','SYNSET1','SYNSET2','WUP_SIM','JCN_SIM','TOKEN_SIM','LIN_SIM'])
print(clusterer.cluster_similarity_model.intercept_[0])
for feature, coef in zip(clusterer.cluster_similarity_model.feature_names_in_,clusterer.cluster_similarity_model.coef_[0]):
    print(feature,coef)

In [ ]:
# Optimizing takes some time and exact reproducability may depend on the versions of the relevant installed Python packages. Instead, I provide the results of one run, but if you want to run the optimizer, use:
# optimizer = clusterer.optimize_cluster_algorithm()
# clusterer.clustering_settings = optimizer.max['params']
with open('clustering/clustering_settings.json',encoding='utf8') as infile:
    clusterer.clustering_settings = json.load(infile)
print(clusterer.clustering_settings)

In [ ]:
# We evaluate a cluster model with the optimal cluster settings. This function returns F1-score.
clusterer.evaluate_cluster_algorithm(**clusterer.clustering_settings)

# Running the clusterer

In [ ]:
# We build a new clusterer object which is a little more lightweight and discards attributes that are not relevant for the training process
clusterer = SynsetClusterer(token_transformer_model=None,greek_ic_file='ic-greek-nouns.dat',cluster_settings_file='clustering/clustering_settings.json',cluster_similarity_model_file='clustering/cluster_similarity_model',consec_output_dir='all_consec_predictions')

In [ ]:
# lemma_synset_data_test is strctured as follows: lemma: synset: tokens(English WSD probability, word alignment score, glaux_id)
lemma_synset_data_test = clusterer.build_prediction_data(alignment_dir='../Word Alignment/predictions_glaux',wsd_prediction_dir='all_consec_predictions')

In [ ]:
# We cluster 10 randomly chosen lemmas
lemmas = ['αἴσθησις','σταθμός','σημεῖον','φιλοσοφία','κρίσις','λύκος','γένος','ὠδίς','ἰσχύς','κοινωνός']
clusterer.cluster_lemmas(lemmas,lemma_synset_data_test,show_progress=False)

In [ ]:
lemmas = set()
with open('greek_nouns.txt',encoding='utf8') as infile:
    lines = infile.readlines()
    for line in lines:
        lemmas.add(line.strip('\n'))
lemmas = [x for x in lemmas if x in lemma_synset_data_test and x in clusterer.lemma_id]

In [ ]:
# We cluster all noun lemmas in GLAUx for which we have input data. min_threshold is the minimum amount of tokens that a cluster should have: any smaller clusters are discarded. 
clusterer.cluster_lemmas(lemmas,lemma_synset_data_test,min_threshold=10,output_dir='auto_training_data_wsd',show_progress=True)